In [42]:
import numpy as np
import pandas as pd

In [43]:
df=pd.read_csv('diabetes.csv')

In [44]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [45]:
# find realtion which feature is affecting the outcome
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


we can see tht blood pressure and skin thickness are not impacting too much we can remove them but let it be

In [46]:
X=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [47]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [48]:
X=scaler.fit_transform(X)

In [49]:
X.shape

(768, 8)

In [50]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [51]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [52]:
model=Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [53]:
model.fit(X_train,y_train,batch_size=32,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4925 - loss: 0.7357 - val_accuracy: 0.5519 - val_loss: 0.7155
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6265 - loss: 0.6500 - val_accuracy: 0.6364 - val_loss: 0.6639
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6710 - loss: 0.6164 - val_accuracy: 0.6818 - val_loss: 0.6273
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7022 - loss: 0.5864 - val_accuracy: 0.7338 - val_loss: 0.5974
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7163 - loss: 0.5706 - val_accuracy: 0.7662 - val_loss: 0.5727
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7343 - loss: 0.5519 - val_accuracy: 0.7792 - val_loss: 0.5517
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7638 - loss: 0.5174 - val_accuracy: 0.7857 - val_loss: 0.5354
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7407 - loss: 0.5272 - val_accuracy: 0.7922 - 

after doing this much we are clue less. we need to decide the no of layer no of nodes,optimizer, activation,loss etc. we can automate all this using keras tuner library by using this we will find the best suitable parametr value of our model.

**
1. how to select appropriate optimizer.
2. no of nodes in a layer.
3. how to select no of layers.
4. all in one model.
**

In [54]:
pip install keras-tuner

In [55]:
import kerastuner as kt

In [56]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def build_model(hp):
    model = Sequential()

    # Input layer
    model.add(Dense(
        units=32,
        activation='relu',
        input_shape=(8,)
    ))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Hyperparameter for optimizer
    optimizer = hp.Choice(
        'optimizer',
        values=['adam', 'sgd', 'rmsprop', 'adadelta']
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [57]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy', max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [58]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [59]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [60]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [61]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [62]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.6625 - loss: 0.5940 - val_accuracy: 0.7013 - val_loss: 0.5854
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6952 - loss: 0.5766 - val_accuracy: 0.7208 - val_loss: 0.5776
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.7162 - loss: 0.5638 - val_accuracy: 0.7273 - val_loss: 0.5709
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7338 - loss: 0.5338 - val_accuracy: 0.7532 - val_loss: 0.5649
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7189 - loss: 0.5539 - val_accuracy: 0.7597 - val_loss: 0.5593
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7193 - loss: 0.5480 - val_accuracy: 0.7597 - val_loss: 0.5543
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7341 - loss: 0.5320 - val_accuracy: 0.7597 - val_loss: 0.5497
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7618 - loss: 0.5224 - val_accurac

In [67]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# right number of neurons
def build_model(hp):
    model = Sequential()

    units = hp.Int(
        'units',
        min_value=8,
        max_value=128,
        step=8
    )

    model.add(Dense(
        units=units,
        activation='relu',
        input_shape=(8,)
    ))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [69]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='mydir',project_name='Diabetes')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [70]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.6753246784210205

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 20s


In [71]:
tuner.get_best_hyperparameters()[0].values

{'units': 88}

In [74]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [75]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.7624 - loss: 0.5629
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.7603 - loss: 0.5117
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7621 - loss: 0.4915
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7677 - loss: 0.4839
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7863 - loss: 0.4640
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7667 - loss: 0.4776
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8012 - loss: 0.4507
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7910 - loss: 0.4452
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7721 - loss: 0.4673
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7437 - loss: 0.4826
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7648 - loss: 0.4747
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - 

In [78]:
# how to select no of layer
def build_model(hp):
  model=Sequential()
  model.add(Dense(88,activation='relu',input_dim=8))
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
    model.add(Dense(88,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model



In [79]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,directory='mydir',project_name='num_layers')

In [80]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 05s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 17s


In [81]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 9}

In [82]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 24 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [83]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.7474 - loss: 0.5010 - val_accuracy: 0.7857 - val_loss: 0.4990
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8025 - loss: 0.4082 - val_accuracy: 0.7403 - val_loss: 0.5741
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7913 - loss: 0.4131 - val_accuracy: 0.7922 - val_loss: 0.4966
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8210 - loss: 0.3797 - val_accuracy: 0.8052 - val_loss: 0.4982
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8422 - loss: 0.3730 - val_accuracy: 0.7857 - val_loss: 0.5262
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8425 - loss: 0.3725 - val_accuracy: 0.7662 - val_loss: 0.6761
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8284 - loss: 0.4163 - val_accuracy: 0.7857 - val_loss: 0.5681
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8718 - loss: 0.3285 - val_accurac

In [95]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

def build_model(hp):
    model = Sequential()

    # Number of hidden layers
    num_layers = hp.Int('num_layers', min_value=1, max_value=10)

    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=8, max_value=128, step=8)
        activation = hp.Choice(f'activation_{i}', ['relu', 'tanh', 'sigmoid'])
        dropout_rate = hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.5, step=0.1)

        if i == 0:
            model.add(Dense(
                units=units,
                activation=activation,
                input_shape=(8,)
            ))
        else:
            model.add(Dense(
                units=units,
                activation=activation
            ))

        model.add(Dropout(dropout_rate))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile model
    model.compile(
        optimizer=hp.Choice(
            'optimizer',
            values=['adam', 'rmsprop', 'nadam', 'sgd']
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [97]:
tuner=kt.RandomSearch(build_model,
                      objective='val_accuracy',
                      max_trials=3,
                      directory='mydir',
                      project_name='final'

                      )

Reloading Tuner from mydir/final/tuner0.json


In [98]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [99]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units_0': 120,
 'activation_0': 'tanh',
 'optimizer': 'nadam',
 'units_1': 8,
 'activation_1': 'relu',
 'units_2': 8,
 'activation_2': 'relu'}

In [100]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 19 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [101]:
model.fit(X_train,y_train,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.7746 - loss: 0.6390 - val_accuracy: 0.7922 - val_loss: 0.6185
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7667 - loss: 0.6413 - val_accuracy: 0.7792 - val_loss: 0.6142
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8078 - loss: 0.6134 - val_accuracy: 0.7662 - val_loss: 0.6096
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7756 - loss: 0.6168 - val_accuracy: 0.7468 - val_loss: 0.5991
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7803 - loss: 0.6084 - val_accuracy: 0.7532 - val_loss: 0.5807
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7876 - loss: 0.5830 - val_accuracy: 0.7597 - val_loss: 0.5546
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7958 - loss: 0.5585 - val_accuracy: 0.7597 - val_loss: 0.5175
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7882 - loss: 0.5075 - val_accuracy: 0.7